## Main processing of the data for exploring word embeddings in the context of Ukrainian-English translation

Libraries to install

In [12]:
import fasttext
import numpy as np

from alignment import (
    load_pairs,
    build_aligned_matrices,
    learn_least_squares,
    learn_orthogonal,
    build_candidate_matrices,
    translate_word,
    evaluate,
)

[FastText](https://fasttext.cc/docs/en/crawl-vectors.html) embedding models for Ukrainian and English

In [13]:
model_uk = fasttext.load_model('../model/cc.uk.300.bin')
model_en = fasttext.load_model('../model/cc.en.300.bin')

Creating pairs of words - Ukrainian and English

In [14]:
train_dict = {}

with open("../data/usage/uk-en-train.csv", "r", encoding="utf-8") as f_train:
    next(f_train)
    for line in f_train:
        line = line.strip("\n")
        ua_word, eng_word = line.split(" ")

        if ua_word not in train_dict:
            train_dict.setdefault(ua_word, [])
        train_dict[ua_word].append(eng_word)

In [15]:
pairs_train = []

for ua, eng_words in train_dict.items():
    for eng in eng_words:
        pairs_train.append((ua, eng))

Building aligned embeddings matrices for Ukrainian and English words

In [16]:
ua_embeddings, eng_embeddings = [], []

for ua, eng in pairs_train:
    ua_embeddings.append(model_uk.get_word_vector(ua))
    eng_embeddings.append(model_en.get_word_vector(eng))

A_ua, B_eng = np.column_stack(ua_embeddings), np.column_stack(eng_embeddings)

Learning tne least Square Mapping and Orthogonal

In [17]:
W_ls = learn_least_squares(A_ua, B_eng)
W_ortho = learn_orthogonal(A_ua, B_eng)
print("Least squares matrix shape:", W_ls.shape)
print("Orthogonal matrix shape:", W_ortho.shape)

Least squares matrix shape: (300, 300)
Orthogonal matrix shape: (300, 300)


Preparing English Candidate words for translation

In [18]:
candidate_words = []
for _, eng_words in train_dict.items():
    for eng in eng_words:
        candidate_words.append(eng)

candidate_words, E = build_candidate_matrices(candidate_words, model_en)
print("Number of candidate words:", len(candidate_words))
print("Candidate matrix shape:", E.shape)

Number of candidate words: 5184
Candidate matrix shape: (5184, 300)


Translating a Ukrainian word

In [19]:
translate_word("кіт", W_ls, model_uk, candidate_words, E, top_k=5)

[('cat', 0.7612910270690918),
 ('tom', 0.5609461069107056),
 ('wolf', 0.5537139177322388),
 ('sophie', 0.5086208581924438),
 ('sam', 0.5041936039924622)]

In [20]:
translate_word("кіт", W_ortho, model_uk, candidate_words, E, top_k=5)

[('cat', 0.7347657680511475),
 ('wolf', 0.48148325085639954),
 ('lion', 0.449465811252594),
 ('tom', 0.44930291175842285),
 ('animal', 0.4349265396595001)]

Evalueting the results

In [21]:
accuracy_ls, results_ls = evaluate(pairs_train, W_ls, model_uk, candidate_words, E, top_k=1)
print("Least squares accuracy:", accuracy_ls)
results_ls[:5]

Least squares accuracy: 0.00014365752047119666


[{'ukrainian': 'категорія',
  'true_english': 'category',
  'predictions': ['category'],
  'correct': True}]

In [22]:
accuracy_ortho, results_ortho = evaluate(pairs_train, W_ortho, model_uk, candidate_words, E, top_k=1)
print("Orthogonal Procrustes accuracy:", accuracy_ortho)
results_ortho[:5]

Orthogonal Procrustes accuracy: 0.00014365752047119666


[{'ukrainian': 'категорія',
  'true_english': 'category',
  'predictions': ['category'],
  'correct': True}]